# Fine-tuning LLMs for Text Summarization

In this assignment you will have to fine-tuned pre-trained Large Language Models for the task of **text summarization**. For this task we will use the [**CNN Dailymail dataset**](https://huggingface.co/datasets/abisee/cnn_dailymail). The CNN DailyMail Dataset is an English-language dataset containing just over 300k unique news articles as written by journalists at CNN and the Daily Mail. For each instance, there is a string for the article and a string for the highlights that form a summary of the article. The dataset has 287,113 samples for training, 13,368 for validation adn 11,490 for testing.

The final goal of this assignment is achieve the best performance in this task by fine-tuning a pre-trained model under 1B parameters. You will have to fine-tune at least one encoder-decoder model and one decoder-only model. You can explore different configurations of the models (architecture, model size, sampling strategy, ...), selection and processing of the training dataset (training size, prompting strategies, context length, ...) and the training process (training recipes, optimization hyperparameters, batch size, number of epochs, ...). Furthermore, pick two or three of these factors and perform a detailed analysis of their impact in the final results.  

At the end write a report describing your final configurations (one for encoder-decoder and one for decoder-only), the different options that you have explored and why, and a detailed analysis of the impact they have in the final results.  


### Preparing the dataset

In [1]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")



In [2]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


## Setup and Installation

In [ ]:
!pip install -q transformers[torch] datasets evaluate rouge_score accelerate sentencepiece

In [ ]:
import os
os.environ["USE_TF"] = "0"  # Prevent transformers from importing broken TensorFlow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import json
import gc
from collections import defaultdict

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    DataCollatorForLanguageModeling,
    GenerationConfig,
)
import evaluate

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CONFIGURATION - Adjust these settings based on your hardware
# ============================================================

# Set True to skip base model training and load from saved checkpoints.
SKIP_BASE_TRAINING = False
CHECKPOINT_BASE_PATH = "/kaggle/input/datasets/santirodriguez14/bart-and-gpt-trained-on-cnn-dailymail-dataset/results"

CONFIG = {
    # Models
    "enc_dec_model": "facebook/bart-base",    # ~139M params (encoder-decoder)
    "decoder_model": "openai-community/gpt2",  # ~124M params (decoder-only)
    
    # Data
    "max_input_length": 512,       # Max tokens for article input
    "max_target_length": 128,      # Max tokens for summary output
    "default_train_size": 50000,   # Default training subset size
    "val_size": 5000,              # Validation subset (for faster eval)
    "test_size": 3000,             # Test subset for final evaluation
    
    # Training defaults
    "batch_size": 8,               # Adjust based on GPU memory
    "gradient_accumulation_steps": 4,
    "num_epochs": 3,
    "learning_rate": 3e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "fp16": torch.cuda.is_available(),
    
    # Generation
    "num_beams": 4,
    "max_gen_length": 128,
    
    # Paths
    "output_dir": "./results",
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nSKIP_BASE_TRAINING: {SKIP_BASE_TRAINING}")
if SKIP_BASE_TRAINING:
    print(f"CHECKPOINT_BASE_PATH: {CHECKPOINT_BASE_PATH}")

## Data Exploration

Let's examine the dataset to understand article and summary lengths, which will inform our preprocessing choices.

In [ ]:
# Explore a few samples
for i in range(3):
    sample = dataset["train"][i]
    print(f"--- Sample {i} ---")
    print(f"Article (first 300 chars): {sample['article'][:300]}...")
    print(f"Highlights: {sample['highlights']}")
    print(f"Article length (chars): {len(sample['article'])}")
    print(f"Highlights length (chars): {len(sample['highlights'])}")
    print()

In [ ]:
# Length statistics (word-level)
train_article_lengths = [len(s["article"].split()) for s in dataset["train"]]
train_summary_lengths = [len(s["highlights"].split()) for s in dataset["train"]]

print("Article length (words):")
print(f"  Mean: {np.mean(train_article_lengths):.0f}")
print(f"  Median: {np.median(train_article_lengths):.0f}")
print(f"  P95: {np.percentile(train_article_lengths, 95):.0f}")
print(f"  Max: {np.max(train_article_lengths)}")

print("\nSummary length (words):")
print(f"  Mean: {np.mean(train_summary_lengths):.0f}")
print(f"  Median: {np.median(train_summary_lengths):.0f}")
print(f"  P95: {np.percentile(train_summary_lengths, 95):.0f}")
print(f"  Max: {np.max(train_summary_lengths)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train_article_lengths, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(np.median(train_article_lengths), color="red", linestyle="--", label="Median")
axes[0].set_title("Article Length Distribution (words)")
axes[0].set_xlabel("Number of words")
axes[0].legend()

axes[1].hist(train_summary_lengths, bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[1].axvline(np.median(train_summary_lengths), color="red", linestyle="--", label="Median")
axes[1].set_title("Summary Length Distribution (words)")
axes[1].set_xlabel("Number of words")
axes[1].legend()

plt.tight_layout()
plt.show()

## Data Preprocessing

We define different preprocessing strategies:
- **BART (encoder-decoder):** The article is the input and the summary is the target label. The model learns to map input to output via cross-attention.
- **GPT-2 (decoder-only):** We concatenate article + summary into a single sequence with a prompt template, and mask the loss on the prompt/article tokens so the model only learns to generate the summary portion.

For the **prompt format ablation**, we define three templates for the decoder-only model:
1. **Simple:** `Article: {article}\nSummary: {summary}`
2. **Instruction:** `Summarize the following news article.\n\nArticle: {article}\n\nSummary: {summary}`
3. **Structured:** `### Article\n{article}\n\n### Summary\n{summary}`

In [ ]:
# ============================================================
# PROMPT TEMPLATES (for decoder-only model ablation)
# ============================================================

PROMPT_TEMPLATES = {
    "simple": {
        "train": "Article: {article}\nSummary: {summary}",
        "infer": "Article: {article}\nSummary:",
    },
    "instruction": {
        "train": "Summarize the following news article.\n\nArticle: {article}\n\nSummary: {summary}",
        "infer": "Summarize the following news article.\n\nArticle: {article}\n\nSummary:",
    },
    "structured": {
        "train": "### Article\n{article}\n\n### Summary\n{summary}",
        "infer": "### Article\n{article}\n\n### Summary\n",
    },
}

print("Defined 3 prompt templates: simple, instruction, structured")

In [ ]:
# ============================================================
# PREPROCESSING: Encoder-Decoder (BART)
# ============================================================

def preprocess_bart(examples, tokenizer, max_input_length, max_target_length):
    """Tokenize articles as inputs and summaries as targets for BART."""
    # Add prefix for T5-style models (BART doesn't strictly need it, but it's harmless)
    inputs = ["summarize: " + article for article in examples["article"]]
    targets = examples["highlights"]
    
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding=False,
    )
    
    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding=False,
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# ============================================================
# PREPROCESSING: Decoder-Only (GPT-2)
# ============================================================

def preprocess_gpt2(examples, tokenizer, max_length, prompt_template="simple"):
    """
    Tokenize for causal LM: concatenate article + summary with prompt template.
    Mask loss on the prompt/article portion so the model only learns to generate summaries.
    """
    template = PROMPT_TEMPLATES[prompt_template]
    
    all_input_ids = []
    all_attention_mask = []
    all_labels = []
    
    for article, summary in zip(examples["article"], examples["highlights"]):
        # Build the inference prompt (everything before the summary)
        prompt_text = template["infer"].format(article=article)
        # Build the full training sequence
        full_text = template["train"].format(article=article, summary=summary) + tokenizer.eos_token
        
        # Tokenize the prompt and full sequence
        prompt_ids = tokenizer(prompt_text, truncation=False, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(full_text, truncation=False, add_special_tokens=False)["input_ids"]
        
        # Truncate if needed: keep as much of the prompt as possible + full summary
        if len(full_ids) > max_length:
            # Calculate how many prompt tokens we can keep
            summary_ids = tokenizer(summary + tokenizer.eos_token, truncation=False, add_special_tokens=False)["input_ids"]
            available_for_prompt = max_length - len(summary_ids)
            if available_for_prompt < 50:  # Skip if article would be too short
                continue
            prompt_ids = prompt_ids[:available_for_prompt]
            full_ids = prompt_ids + summary_ids
            full_ids = full_ids[:max_length]
        
        # Create labels: -100 for prompt tokens (no loss), actual ids for summary tokens
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        # Ensure same length
        labels = labels[:len(full_ids)]
        
        all_input_ids.append(full_ids)
        all_attention_mask.append([1] * len(full_ids))
        all_labels.append(labels)
    
    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    }


print("Preprocessing functions defined.")

In [ ]:
# ============================================================
# DATA SUBSET CREATION
# ============================================================

def create_subsets(dataset, train_size, val_size, test_size, seed=42):
    """Create smaller subsets for faster experimentation."""
    train_sub = dataset["train"].shuffle(seed=seed).select(range(min(train_size, len(dataset["train"]))))
    val_sub = dataset["validation"].shuffle(seed=seed).select(range(min(val_size, len(dataset["validation"]))))
    test_sub = dataset["test"].shuffle(seed=seed).select(range(min(test_size, len(dataset["test"]))))
    return train_sub, val_sub, test_sub

# Create default subsets
train_data, val_data, test_data = create_subsets(
    dataset,
    CONFIG["default_train_size"],
    CONFIG["val_size"],
    CONFIG["test_size"],
)

print(f"Training subset: {len(train_data)} samples")
print(f"Validation subset: {len(val_data)} samples")
print(f"Test subset: {len(test_data)} samples")

## Evaluation Metrics

We use ROUGE (Recall-Oriented Understudy for Gisting Evaluation) scores:
- **ROUGE-1:** Unigram overlap between generated and reference summaries
- **ROUGE-2:** Bigram overlap
- **ROUGE-L:** Longest common subsequence

In [ ]:
# ============================================================
# EVALUATION UTILITIES
# ============================================================

rouge_metric = evaluate.load("rouge")

def compute_rouge(predictions, references):
    """Compute ROUGE scores between predictions and references."""
    results = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,
    )
    return {
        "rouge1": results["rouge1"] * 100,
        "rouge2": results["rouge2"] * 100,
        "rougeL": results["rougeL"] * 100,
    }


def compute_metrics_bart(eval_preds, tokenizer):
    """Compute metrics for Seq2Seq trainer (BART)."""
    preds, labels = eval_preds
    # Decode predictions
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Replace -100 with pad token id
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    return compute_rouge(decoded_preds, decoded_labels)


def generate_summaries_bart(model, tokenizer, test_dataset, num_samples=None, batch_size=16):
    """Generate summaries using BART model."""
    model.eval()
    if num_samples:
        test_dataset = test_dataset.select(range(min(num_samples, len(test_dataset))))
    
    predictions = []
    references = [s["highlights"] for s in test_dataset]
    
    for i in range(0, len(test_dataset), batch_size):
        batch_articles = [
            "summarize: " + test_dataset[j]["article"]
            for j in range(i, min(i + batch_size, len(test_dataset)))
        ]
        inputs = tokenizer(
            batch_articles,
            max_length=CONFIG["max_input_length"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=CONFIG["max_gen_length"],
                num_beams=CONFIG["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded)
    
    return predictions, references


def generate_summaries_gpt2(model, tokenizer, test_dataset, prompt_template="simple",
                             num_samples=None, batch_size=8):
    """Generate summaries using GPT-2 model."""
    model.eval()
    template = PROMPT_TEMPLATES[prompt_template]
    
    if num_samples:
        test_dataset = test_dataset.select(range(min(num_samples, len(test_dataset))))
    
    predictions = []
    references = [s["highlights"] for s in test_dataset]
    
    for i in range(0, len(test_dataset), batch_size):
        batch_articles = []
        for j in range(i, min(i + batch_size, len(test_dataset))):
            prompt = template["infer"].format(article=test_dataset[j]["article"])
            batch_articles.append(prompt)
        
        inputs = tokenizer(
            batch_articles,
            max_length=CONFIG["max_input_length"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        
        input_length = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=CONFIG["max_gen_length"],
                num_beams=CONFIG["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        # Only decode the newly generated tokens
        for output in outputs:
            generated = output[input_length:]
            decoded = tokenizer.decode(generated, skip_special_tokens=True)
            predictions.append(decoded.strip())
    
    return predictions, references


print("Evaluation utilities defined.")

## Part 1: Encoder-Decoder Model — BART-base

BART (Bidirectional and Auto-Regressive Transformers) is a denoising autoencoder for pretraining sequence-to-sequence models. BART-base has ~139M parameters and is well-suited for summarization tasks. It uses a standard Transformer encoder-decoder architecture where the encoder processes the article and the decoder generates the summary.

In [ ]:
# ============================================================
# BART-BASE: Load model and tokenizer
# ============================================================

bart_tokenizer = AutoTokenizer.from_pretrained(CONFIG["enc_dec_model"])
bart_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["enc_dec_model"])

# Print model size
num_params = sum(p.numel() for p in bart_model.parameters())
print(f"BART-base parameters: {num_params:,} ({num_params / 1e6:.1f}M)")
print(f"Vocabulary size: {bart_tokenizer.vocab_size}")

In [ ]:
# ============================================================
# BART-BASE: Tokenize datasets
# ============================================================

bart_train_tokenized = train_data.map(
    lambda x: preprocess_bart(x, bart_tokenizer, CONFIG["max_input_length"], CONFIG["max_target_length"]),
    batched=True,
    remove_columns=train_data.column_names,
    desc="Tokenizing BART train",
)

bart_val_tokenized = val_data.map(
    lambda x: preprocess_bart(x, bart_tokenizer, CONFIG["max_input_length"], CONFIG["max_target_length"]),
    batched=True,
    remove_columns=val_data.column_names,
    desc="Tokenizing BART val",
)

print(f"BART tokenized train: {len(bart_train_tokenized)} samples")
print(f"BART tokenized val: {len(bart_val_tokenized)} samples")
print(f"Example input length: {len(bart_train_tokenized[0]['input_ids'])} tokens")
print(f"Example target length: {len(bart_train_tokenized[0]['labels'])} tokens")

In [ ]:
# ============================================================
# BART-BASE: Training (or load from checkpoint)
# ============================================================

import glob as _glob
from functools import partial

if not SKIP_BASE_TRAINING:
    bart_training_args = Seq2SeqTrainingArguments(
        output_dir=os.path.join(CONFIG["output_dir"], "bart-base"),
        eval_strategy="steps",
        eval_steps=2000,
        save_strategy="steps",
        save_steps=2000,
        learning_rate=CONFIG["learning_rate"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=CONFIG["num_epochs"],
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=CONFIG["weight_decay"],
        fp16=CONFIG["fp16"],
        predict_with_generate=False,
        logging_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to="none",
    )

    bart_data_collator = DataCollatorForSeq2Seq(
        tokenizer=bart_tokenizer,
        model=bart_model,
        padding=True,
    )

    bart_trainer = Seq2SeqTrainer(
        model=bart_model,
        args=bart_training_args,
        train_dataset=bart_train_tokenized,
        eval_dataset=bart_val_tokenized,
        processing_class=bart_tokenizer,
        data_collator=bart_data_collator,
    )

    print("BART trainer configured. Starting training...")
    bart_train_result = bart_trainer.train()
    print(f"\nBART training complete!")
    print(f"Training loss: {bart_train_result.training_loss:.4f}")

else:
    # Load from the highest-numbered checkpoint saved by the previous run.
    _ckpt_dir = os.path.join(CHECKPOINT_BASE_PATH, "bart-base")
    _ckpts = sorted(
        _glob.glob(os.path.join(_ckpt_dir, "checkpoint-*")),
        key=lambda p: int(p.rstrip("/\\").split("-")[-1]),
    )
    if not _ckpts:
        raise FileNotFoundError(
            f"No checkpoints found at {_ckpt_dir}\n"
            f"Contents: {os.listdir(_ckpt_dir) if os.path.isdir(_ckpt_dir) else 'directory missing'}"
        )
    _best = _ckpts[-1]
    print(f"SKIP_BASE_TRAINING=True — loading BART from {_best}")
    bart_model = AutoModelForSeq2SeqLM.from_pretrained(_best, local_files_only=True)
    bart_model = bart_model.to(device)
    print("BART loaded.")

In [ ]:
# ============================================================
# BART-BASE: Evaluate on test set
# ============================================================

print("Generating summaries on test set with BART...")
bart_preds, bart_refs = generate_summaries_bart(
    bart_model, bart_tokenizer, test_data, batch_size=8
)

bart_test_scores = compute_rouge(bart_preds, bart_refs)
print(f"\nBART-base Test Results:")
print(f"  ROUGE-1: {bart_test_scores['rouge1']:.2f}")
print(f"  ROUGE-2: {bart_test_scores['rouge2']:.2f}")
print(f"  ROUGE-L: {bart_test_scores['rougeL']:.2f}")

# Show a few example predictions
print("\n" + "="*60)
print("EXAMPLE PREDICTIONS")
print("="*60)
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"Reference: {bart_refs[i][:200]}...")
    print(f"Predicted: {bart_preds[i][:200]}...")

## Part 2: Decoder-Only Model — GPT-2

GPT-2 is a decoder-only transformer with ~124M parameters. For summarization, we frame the task as conditional text generation: given an article as context, generate its summary. During training, we concatenate article + summary and mask the loss on the article tokens so the model only learns to predict the summary portion. We use the "instruction" prompt template as the default.

In [ ]:
# ============================================================
# GPT-2: Load model and tokenizer
# ============================================================

gpt2_tokenizer = AutoTokenizer.from_pretrained(CONFIG["decoder_model"])
gpt2_model = AutoModelForCausalLM.from_pretrained(CONFIG["decoder_model"])

# GPT-2 doesn't have a pad token by default; set it to eos_token
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.config.pad_token_id = gpt2_tokenizer.eos_token_id

# Set padding side to left for generation (important for batch generation)
gpt2_tokenizer.padding_side = "left"

num_params = sum(p.numel() for p in gpt2_model.parameters())
print(f"GPT-2 parameters: {num_params:,} ({num_params / 1e6:.1f}M)")
print(f"Vocabulary size: {gpt2_tokenizer.vocab_size}")
print(f"Max position embeddings: {gpt2_model.config.n_positions}")

In [ ]:
# ============================================================
# GPT-2: Custom data collator with padding for variable-length sequences
# ============================================================

class GPT2SummarizationCollator:
    """Custom collator that pads input_ids, attention_mask, and labels."""
    
    def __init__(self, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __call__(self, features):
        # Find max length in this batch
        max_len = min(
            max(len(f["input_ids"]) for f in features),
            self.max_length,
        )
        
        batch = {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }
        
        for f in features:
            input_ids = f["input_ids"][:max_len]
            attention_mask = f["attention_mask"][:max_len]
            labels = f["labels"][:max_len]
            
            # Pad from the right
            pad_len = max_len - len(input_ids)
            input_ids = input_ids + [self.tokenizer.pad_token_id] * pad_len
            attention_mask = attention_mask + [0] * pad_len
            labels = labels + [-100] * pad_len
            
            batch["input_ids"].append(input_ids)
            batch["attention_mask"].append(attention_mask)
            batch["labels"].append(labels)
        
        batch = {k: torch.tensor(v) for k, v in batch.items()}
        return batch

print("Custom GPT-2 collator defined.")

In [ ]:
# ============================================================
# GPT-2: Tokenize datasets (using "instruction" template by default)
# ============================================================

# GPT-2 max context is 1024 tokens
gpt2_max_length = min(CONFIG["max_input_length"] + CONFIG["max_target_length"], 1024)

gpt2_train_tokenized = train_data.map(
    lambda x: preprocess_gpt2(x, gpt2_tokenizer, gpt2_max_length, prompt_template="instruction"),
    batched=True,
    remove_columns=train_data.column_names,
    desc="Tokenizing GPT-2 train",
)

gpt2_val_tokenized = val_data.map(
    lambda x: preprocess_gpt2(x, gpt2_tokenizer, gpt2_max_length, prompt_template="instruction"),
    batched=True,
    remove_columns=val_data.column_names,
    desc="Tokenizing GPT-2 val",
)

print(f"GPT-2 tokenized train: {len(gpt2_train_tokenized)} samples")
print(f"GPT-2 tokenized val: {len(gpt2_val_tokenized)} samples")
print(f"Example sequence length: {len(gpt2_train_tokenized[0]['input_ids'])} tokens")

In [ ]:
# ============================================================
# GPT-2: Training (or load from checkpoint)
# ============================================================

if not SKIP_BASE_TRAINING:
    gpt2_training_args = TrainingArguments(
        output_dir=os.path.join(CONFIG["output_dir"], "gpt2"),
        eval_strategy="steps",
        eval_steps=2000,
        save_strategy="steps",
        save_steps=2000,
        learning_rate=CONFIG["learning_rate"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=CONFIG["num_epochs"],
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=CONFIG["weight_decay"],
        fp16=CONFIG["fp16"],
        logging_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to="none",
    )

    gpt2_collator = GPT2SummarizationCollator(gpt2_tokenizer, max_length=gpt2_max_length)

    gpt2_trainer = Trainer(
        model=gpt2_model,
        args=gpt2_training_args,
        train_dataset=gpt2_train_tokenized,
        eval_dataset=gpt2_val_tokenized,
        processing_class=gpt2_tokenizer,
        data_collator=gpt2_collator,
    )

    print("GPT-2 trainer configured. Starting training...")
    gpt2_train_result = gpt2_trainer.train()
    print(f"\nGPT-2 training complete!")
    print(f"Training loss: {gpt2_train_result.training_loss:.4f}")

else:
    _ckpt_dir = os.path.join(CHECKPOINT_BASE_PATH, "gpt2")
    _ckpts = sorted(
        _glob.glob(os.path.join(_ckpt_dir, "checkpoint-*")),
        key=lambda p: int(p.rstrip("/\\").split("-")[-1]),
    )
    if not _ckpts:
        raise FileNotFoundError(
            f"No checkpoints found at {_ckpt_dir}\n"
            f"Contents: {os.listdir(_ckpt_dir) if os.path.isdir(_ckpt_dir) else 'directory missing'}"
        )
    _best = _ckpts[-1]
    print(f"SKIP_BASE_TRAINING=True — loading GPT-2 from {_best}")
    gpt2_model = AutoModelForCausalLM.from_pretrained(_best, local_files_only=True)
    gpt2_model = gpt2_model.to(device)
    print("GPT-2 loaded.")

In [ ]:
# ============================================================
# GPT-2: Evaluate on test set
# ============================================================

print("Generating summaries on test set with GPT-2...")
gpt2_preds, gpt2_refs = generate_summaries_gpt2(
    gpt2_model, gpt2_tokenizer, test_data, prompt_template="instruction", batch_size=4
)

gpt2_test_scores = compute_rouge(gpt2_preds, gpt2_refs)
print(f"\nGPT-2 Test Results:")
print(f"  ROUGE-1: {gpt2_test_scores['rouge1']:.2f}")
print(f"  ROUGE-2: {gpt2_test_scores['rouge2']:.2f}")
print(f"  ROUGE-L: {gpt2_test_scores['rougeL']:.2f}")

# Show examples
print("\n" + "="*60)
print("EXAMPLE PREDICTIONS")
print("="*60)
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"Reference: {gpt2_refs[i][:200]}...")
    print(f"Predicted: {gpt2_preds[i][:200]}...")

## Part 3: Ablation Studies

We analyze the impact of three key factors on summarization performance:

1. **Training Dataset Size:** How does the amount of training data affect ROUGE scores? We compare 5k, 15k, and 50k training samples.
2. **Learning Rate:** We compare three learning rates (1e-5, 3e-5, 5e-5) to study the sensitivity of each model architecture to this hyperparameter.
3. **Prompt Format (decoder-only):** We compare three different prompt templates to understand how the framing of the task affects GPT-2's summarization ability.

For efficiency, ablation experiments use a smaller evaluation set (1000 test samples) and train for 2 epochs.

In [ ]:
# ============================================================
# FREE BASE MODEL MEMORY BEFORE ABLATIONS
# Ablations each load their own fresh model instances, so the
# base bart_model and gpt2_model can be released now.
# ============================================================

del bart_model, gpt2_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_reserved(0)) / 1e9
    print(f"Base models freed. ~{free_gb:.1f} GB available on GPU.")
else:
    print("Base models freed.")

In [ ]:
# ============================================================
# ABLATION HELPER: Train and evaluate a model configuration
# ============================================================

# Store all ablation results
ablation_results = defaultdict(list)

ABLATION_TEST_SIZE = 1000  # Smaller test set for ablation speed
ABLATION_EPOCHS = 2

def run_bart_ablation(train_subset, val_subset, test_subset, lr, run_name, epochs=ABLATION_EPOCHS):
    """Train BART with given config and return ROUGE scores."""
    print(f"\n{'='*60}")
    print(f"BART Ablation: {run_name}")
    print(f"  Train size: {len(train_subset)}, LR: {lr}, Epochs: {epochs}")
    print(f"{'='*60}")
    
    # Fresh model
    model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["enc_dec_model"])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["enc_dec_model"])
    
    # Tokenize
    train_tok = train_subset.map(
        lambda x: preprocess_bart(x, tokenizer, CONFIG["max_input_length"], CONFIG["max_target_length"]),
        batched=True, remove_columns=train_subset.column_names,
    )
    val_tok = val_subset.map(
        lambda x: preprocess_bart(x, tokenizer, CONFIG["max_input_length"], CONFIG["max_target_length"]),
        batched=True, remove_columns=val_subset.column_names,
    )
    
    # NOTE: predict_with_generate=False — beam-search eval at every checkpoint
    # is the dominant runtime cost and the cause of previous hangs.
    args = Seq2SeqTrainingArguments(
        output_dir=os.path.join(CONFIG["output_dir"], f"ablation/{run_name}"),
        eval_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=epochs,
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=CONFIG["weight_decay"],
        fp16=CONFIG["fp16"],
        predict_with_generate=False,
        logging_steps=500,
        save_strategy="no",
        report_to="none",
    )
    
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_tok, eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
    )
    
    trainer.train()
    
    # Evaluate
    preds, refs = generate_summaries_bart(model, tokenizer, test_subset, batch_size=8)
    scores = compute_rouge(preds, refs)
    
    print(f"  Results: R1={scores['rouge1']:.2f}, R2={scores['rouge2']:.2f}, RL={scores['rougeL']:.2f}")
    
    # Clean up GPU memory
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    return scores


def run_gpt2_ablation(train_subset, val_subset, test_subset, lr, prompt_template, 
                       run_name, epochs=ABLATION_EPOCHS):
    """Train GPT-2 with given config and return ROUGE scores."""
    print(f"\n{'='*60}")
    print(f"GPT-2 Ablation: {run_name}")
    print(f"  Train size: {len(train_subset)}, LR: {lr}, Prompt: {prompt_template}, Epochs: {epochs}")
    print(f"{'='*60}")
    
    # Fresh model
    model = AutoModelForCausalLM.from_pretrained(CONFIG["decoder_model"])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["decoder_model"])
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    
    max_len = min(CONFIG["max_input_length"] + CONFIG["max_target_length"], 1024)
    
    # Tokenize
    train_tok = train_subset.map(
        lambda x: preprocess_gpt2(x, tokenizer, max_len, prompt_template=prompt_template),
        batched=True, remove_columns=train_subset.column_names,
    )
    val_tok = val_subset.map(
        lambda x: preprocess_gpt2(x, tokenizer, max_len, prompt_template=prompt_template),
        batched=True, remove_columns=val_subset.column_names,
    )
    
    args = TrainingArguments(
        output_dir=os.path.join(CONFIG["output_dir"], f"ablation/{run_name}"),
        eval_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=epochs,
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=CONFIG["weight_decay"],
        fp16=CONFIG["fp16"],
        logging_steps=500,
        save_strategy="no",
        report_to="none",
    )
    
    collator = GPT2SummarizationCollator(tokenizer, max_length=max_len)
    
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_tok, eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=collator,
    )
    
    trainer.train()
    
    # Evaluate
    preds, refs = generate_summaries_gpt2(
        model, tokenizer, test_subset, prompt_template=prompt_template, batch_size=4
    )
    scores = compute_rouge(preds, refs)
    
    print(f"  Results: R1={scores['rouge1']:.2f}, R2={scores['rouge2']:.2f}, RL={scores['rougeL']:.2f}")
    
    # Clean up
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    return scores


print("Ablation helper functions defined.")

### Ablation 1: Training Dataset Size

We investigate how the number of training samples affects model performance. We test with 5,000, 15,000, and 50,000 samples using the default learning rate (3e-5).

In [ ]:
# ============================================================
# ABLATION 1: Training Dataset Size
# ============================================================

# Trimmed: 5k -> 15k -> 30k (dropped 50k — same insight, ~40% less compute).
train_sizes = [5000, 15000, 30000]
ablation_lr = 3e-5

# Smaller val set for fast loss-based eval during ablation training.
val_abl, test_abl = val_data.select(range(500)), test_data.select(range(ABLATION_TEST_SIZE))

print("=" * 60)
print("ABLATION 1: Training Dataset Size")
print("=" * 60)

for size in train_sizes:
    train_sub = dataset["train"].shuffle(seed=42).select(range(size))
    
    # BART
    scores = run_bart_ablation(
        train_sub, val_abl, test_abl,
        lr=ablation_lr,
        run_name=f"bart_size_{size}",
    )
    ablation_results["train_size"].append({
        "model": "BART-base", "train_size": size, **scores
    })
    
    # GPT-2
    scores = run_gpt2_ablation(
        train_sub, val_abl, test_abl,
        lr=ablation_lr,
        prompt_template="instruction",
        run_name=f"gpt2_size_{size}",
    )
    ablation_results["train_size"].append({
        "model": "GPT-2", "train_size": size, **scores
    })

print("\nTraining size ablation complete!")
pd.DataFrame(ablation_results["train_size"])

### Ablation 2: Learning Rate

We compare three learning rates (1e-5, 3e-5, 5e-5) using a fixed training size of 15,000 samples to isolate the effect of learning rate on convergence and final performance.

In [ ]:
# ============================================================
# ABLATION 2: Learning Rate
# ============================================================

learning_rates = [1e-5, 3e-5, 5e-5]
ablation_train_size = 15000
train_sub_lr = dataset["train"].shuffle(seed=42).select(range(ablation_train_size))

print("=" * 60)
print("ABLATION 2: Learning Rate")
print("=" * 60)

for lr in learning_rates:
    # BART
    scores = run_bart_ablation(
        train_sub_lr, val_abl, test_abl,
        lr=lr,
        run_name=f"bart_lr_{lr}",
    )
    ablation_results["learning_rate"].append({
        "model": "BART-base", "learning_rate": lr, **scores
    })
    
    # GPT-2
    scores = run_gpt2_ablation(
        train_sub_lr, val_abl, test_abl,
        lr=lr,
        prompt_template="instruction",
        run_name=f"gpt2_lr_{lr}",
    )
    ablation_results["learning_rate"].append({
        "model": "GPT-2", "learning_rate": lr, **scores
    })

print("\nLearning rate ablation complete!")
pd.DataFrame(ablation_results["learning_rate"])

### Ablation 3: Prompt Format (Decoder-Only)

We test three prompt templates on GPT-2 to understand how task framing affects performance. This ablation is specific to decoder-only models since encoder-decoder models use a fixed input-output format.

- **Simple:** Minimal prefix (`Article: ... Summary: ...`)
- **Instruction:** Explicit task instruction before the article
- **Structured:** Markdown-style section headers

In [ ]:
# ============================================================
# ABLATION 3: Prompt Format (GPT-2 only)
# ============================================================

prompt_formats = ["simple", "instruction", "structured"]

print("=" * 60)
print("ABLATION 3: Prompt Format (GPT-2)")
print("=" * 60)

for fmt in prompt_formats:
    scores = run_gpt2_ablation(
        train_sub_lr, val_abl, test_abl,
        lr=3e-5,
        prompt_template=fmt,
        run_name=f"gpt2_prompt_{fmt}",
    )
    ablation_results["prompt_format"].append({
        "model": "GPT-2", "prompt_format": fmt, **scores
    })

print("\nPrompt format ablation complete!")
pd.DataFrame(ablation_results["prompt_format"])

## Part 4: Results Visualization and Analysis

In [ ]:
# ============================================================
# COMPARISON: BART vs GPT-2 Final Results
# ============================================================

final_results = pd.DataFrame([
    {"Model": "BART-base (enc-dec)", **bart_test_scores},
    {"Model": "GPT-2 (dec-only)", **gpt2_test_scores},
])

print("=" * 60)
print("FINAL MODEL COMPARISON (Test Set)")
print("=" * 60)
print(final_results.to_string(index=False))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
width = 0.35
metrics = ["rouge1", "rouge2", "rougeL"]
metric_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]

bart_scores_list = [bart_test_scores[m] for m in metrics]
gpt2_scores_list = [gpt2_test_scores[m] for m in metrics]

bars1 = ax.bar(x - width/2, bart_scores_list, width, label="BART-base", color="#4C72B0")
bars2 = ax.bar(x + width/2, gpt2_scores_list, width, label="GPT-2", color="#DD8452")

ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_title("BART-base vs GPT-2: ROUGE Scores on Test Set")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend()
ax.bar_label(bars1, fmt="%.1f", padding=3)
ax.bar_label(bars2, fmt="%.1f", padding=3)
ax.set_ylim(0, max(bart_scores_list + gpt2_scores_list) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: Ablation 1 — Training Size
# ============================================================

df_size = pd.DataFrame(ablation_results["train_size"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, metric in enumerate(["rouge1", "rouge2", "rougeL"]):
    for model_name in ["BART-base", "GPT-2"]:
        model_data = df_size[df_size["model"] == model_name]
        axes[idx].plot(
            model_data["train_size"], model_data[metric],
            marker="o", linewidth=2, label=model_name,
        )
    axes[idx].set_xlabel("Training Samples")
    axes[idx].set_ylabel("Score")
    axes[idx].set_title(f"{metric.upper()} vs Training Size")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Ablation 1: Effect of Training Dataset Size", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nDetailed results:")
print(df_size.to_string(index=False))

In [ ]:
# ============================================================
# VISUALIZATION: Ablation 2 — Learning Rate
# ============================================================

df_lr = pd.DataFrame(ablation_results["learning_rate"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, metric in enumerate(["rouge1", "rouge2", "rougeL"]):
    for model_name in ["BART-base", "GPT-2"]:
        model_data = df_lr[df_lr["model"] == model_name]
        axes[idx].plot(
            [str(lr) for lr in model_data["learning_rate"]],
            model_data[metric],
            marker="s", linewidth=2, label=model_name,
        )
    axes[idx].set_xlabel("Learning Rate")
    axes[idx].set_ylabel("Score")
    axes[idx].set_title(f"{metric.upper()} vs Learning Rate")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].tick_params(axis="x", rotation=15)

plt.suptitle("Ablation 2: Effect of Learning Rate", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nDetailed results:")
print(df_lr.to_string(index=False))

In [ ]:
# ============================================================
# VISUALIZATION: Ablation 3 — Prompt Format
# ============================================================

df_prompt = pd.DataFrame(ablation_results["prompt_format"])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(prompt_formats))
width = 0.25

for idx, metric in enumerate(["rouge1", "rouge2", "rougeL"]):
    values = [df_prompt[df_prompt["prompt_format"] == fmt][metric].values[0] for fmt in prompt_formats]
    bars = ax.bar(x + idx * width, values, width, label=metric.upper())
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=8)

ax.set_xlabel("Prompt Format")
ax.set_ylabel("Score")
ax.set_title("Ablation 3: Effect of Prompt Format on GPT-2 Performance")
ax.set_xticks(x + width)
ax.set_xticklabels([f.capitalize() for f in prompt_formats])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nDetailed results:")
print(df_prompt.to_string(index=False))

In [ ]:
# ============================================================
# SAVE ALL RESULTS TO JSON
# ============================================================

all_results = {
    "final_comparison": {
        "bart_base": bart_test_scores,
        "gpt2": gpt2_test_scores,
    },
    "ablation_train_size": ablation_results["train_size"],
    "ablation_learning_rate": ablation_results["learning_rate"],
    "ablation_prompt_format": ablation_results["prompt_format"],
    "config": {k: str(v) for k, v in CONFIG.items()},
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
with open(os.path.join(CONFIG["output_dir"], "all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to {CONFIG['output_dir']}/all_results.json")

## Report: Fine-tuning LLMs for Text Summarization

### 1. Model Selection and Configuration

**Encoder-Decoder: BART-base (facebook/bart-base)**
- Parameters: ~139M (well under 1B limit)
- Architecture: 6 encoder layers, 6 decoder layers, 768 hidden dimensions
- Why BART: Pre-trained with a denoising objective (text infilling and sentence permutation) that naturally suits generative tasks like summarization. The encoder-decoder architecture allows separate encoding of the full article and autoregressive decoding of the summary, making it architecturally ideal for conditional generation.
- Input processing: Articles prefixed with "summarize:" and tokenized to max 512 tokens; summaries tokenized to max 128 tokens as labels.

**Decoder-Only: GPT-2 (openai-community/gpt2)**
- Parameters: ~124M (well under 1B limit)
- Architecture: 12 transformer decoder layers, 768 hidden dimensions, 1024 max position embeddings
- Why GPT-2: A foundational decoder-only model that demonstrates how causal language models can be adapted for conditional generation tasks. Unlike encoder-decoder models, GPT-2 must process both input and output in a single left-to-right pass, requiring careful prompt engineering and loss masking.
- Input processing: Article and summary concatenated with a prompt template; loss masked on article tokens so the model only learns to generate the summary portion.

**Common Training Configuration:**
- Optimizer: AdamW with weight decay 0.01
- Batch size: 8 per device with 4 gradient accumulation steps (effective batch size 32)
- Warmup: 10% of total steps with linear schedule
- Mixed precision (FP16) when GPU is available
- Generation: Beam search with 4 beams, no-repeat 3-gram constraint

---

### 2. Ablation Study Analysis

#### 2.1 Training Dataset Size (5k vs 15k vs 50k samples)

**Hypothesis:** More training data should improve performance, but with diminishing returns as the models are relatively small.

**Expected findings:**
- Both models should show clear improvements from 5k to 15k, with smaller gains from 15k to 50k
- BART should be more data-efficient due to its pre-training objective being closer to the summarization task
- GPT-2 may require more data to learn the summarization format since it was pre-trained for general language modeling

**Analysis framework:** The curves plotted above show how ROUGE scores scale with training data. Look for where diminishing returns set in — this is the "sweet spot" for practical training. If BART plateaus earlier than GPT-2, it suggests the encoder-decoder architecture provides a stronger inductive bias for summarization.

#### 2.2 Learning Rate (1e-5 vs 3e-5 vs 5e-5)

**Hypothesis:** There exists an optimal learning rate that balances learning speed with training stability. Too low may underfit; too high may cause catastrophic forgetting of pre-trained knowledge.

**Expected findings:**
- BART typically performs best at 3e-5 for fine-tuning (consistent with BERT-family recommendations)
- GPT-2 may prefer a slightly lower rate (1e-5 to 3e-5) to avoid destroying pre-trained representations
- Higher learning rates (5e-5) may cause instability, especially for GPT-2

**Analysis framework:** The learning rate plots reveal each model's sensitivity to this hyperparameter. A flat curve means the model is robust; a sharp peak means careful tuning is critical. Compare the variance across learning rates between the two architectures.

#### 2.3 Prompt Format (Simple vs Instruction vs Structured)

**Hypothesis:** The prompt format affects how well GPT-2 can distinguish the article context from the summary generation task.

**Expected findings:**
- The "instruction" format likely performs best because it explicitly states the task, helping the model understand what is expected
- The "simple" format may perform surprisingly well since it's concise and leaves more token budget for the article
- The "structured" format with markdown headers may help or hurt depending on how much GPT-2 learned about markdown during pre-training

**Analysis framework:** Since this ablation is GPT-2-specific, it highlights a key difference between encoder-decoder and decoder-only models: decoder-only models depend heavily on how the task is framed in the prompt. The results show which framing best activates GPT-2's summarization capabilities.

---

### 3. Key Takeaways

**Encoder-decoder vs Decoder-only:** BART-base is expected to outperform GPT-2 on summarization because its architecture is specifically designed for sequence-to-sequence tasks. The separate encoder allows full bidirectional attention over the article, while the decoder can focus entirely on generation. GPT-2 must divide its fixed context window between reading the article and generating the summary.

**Practical considerations:**
- BART-base provides a better quality-to-cost ratio for summarization-specific tasks
- GPT-2's flexibility (same architecture for many tasks) makes it appealing for multi-task scenarios
- Training size matters: even 15k samples can yield reasonable results with these pre-trained models
- Learning rate selection is critical — a factor of 5x in either direction can significantly affect results
- Prompt engineering for decoder-only models is a form of architecture design that deserves careful attention

---

*Fill in specific numbers from the ablation results above after running the experiments.*